# ThoraxShift runner (Kaggle / any CUDA machine)

Clone -> install -> run -> save. **Nothing scientific happens in this notebook.** Every computation
is a repository script at a pinned, pushed commit, and every run directory records that commit,
its environment and its outputs.

Before running: Accelerator = GPU (T4 x2 or P100), Internet = on, and the data package attached as
an input dataset. Use **Save Version -> Save & Run All** so it runs in the background.

In [ ]:
# ---- configuration: the only cell you edit --------------------------------------------------
REPO = "https://github.com/Sandeepchataut/ThoraxShift.git"
COMMIT = "REPLACE_WITH_FULL_PUSHED_COMMIT_SHA"
DATA = "/kaggle/input/thoraxshift-data"     # contains manifests/, prepared/, masks/
RUNS = "/kaggle/working/runs"               # saved as notebook output
# Run directories from a previous version's output, copied into RUNS before running.
# Then add --resume {RUNS}/<run_id> to the matching command below.
RESUME_FROM = []                            # e.g. ["/kaggle/input/thoraxshift-runner-v3/runs/2026..._deep_..."]
# One command per line. Commands run in parallel, one per GPU (round-robin).
COMMANDS = [
    "scripts/run_deep.py --arch densenet121 --mask lung --protocol indomain --dataset shenzhen",
]

In [ ]:
import os, shutil, subprocess, sys, time
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
SRC = "/kaggle/working/ThoraxShift"
if not os.path.exists(SRC):
    subprocess.run(["git", "clone", "--quiet", REPO, SRC], check=True)
subprocess.run(["git", "-C", SRC, "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", SRC, "checkout", "--quiet", COMMIT], check=True)
head = subprocess.run(["git", "-C", SRC, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert head == COMMIT, f"HEAD {head} != COMMIT {COMMIT} (use the full 40-character SHA)"
print("code at", head)

In [ ]:
# Clean virtual environment with the pinned direct dependencies from pyproject.toml.
VENV = "/kaggle/working/venv"
if not os.path.exists(VENV):
    subprocess.run([sys.executable, "-m", "venv", VENV], check=True)
PY = f"{VENV}/bin/python"
subprocess.run([PY, "-m", "pip", "install", "--quiet", "--upgrade", "pip"], check=True)
subprocess.run([PY, "-m", "pip", "install", "--quiet", "-e", SRC], check=True)
subprocess.run([PY, "-c", "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"], check=True)
dirty = subprocess.run(["git", "-C", SRC, "status", "--porcelain"], capture_output=True, text=True).stdout
assert not dirty.strip(), "working tree is dirty:\n" + dirty

In [ ]:
os.makedirs(RUNS, exist_ok=True)
for src in RESUME_FROM:
    dst = os.path.join(RUNS, os.path.basename(src.rstrip("/")))
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
        print("restored", dst)

In [ ]:
n_gpu = max(1, int(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.count("GPU ")))
env_base = {**os.environ, "TBSHIFT_DATA": DATA, "TBSHIFT_RUNS": RUNS, "PYTHONUNBUFFERED": "1"}
os.makedirs(f"{RUNS}/_logs", exist_ok=True)
pending, running, results = list(enumerate(COMMANDS)), {}, {}
while pending or running:
    free = [g for g in range(n_gpu) if g not in running.values()]
    while pending and free:
        i, cmd = pending.pop(0); g = free.pop(0)
        log = open(f"{RUNS}/_logs/cmd{i:02d}_{time.strftime('%Y%m%dT%H%M%S')}.log", "w")
        p = subprocess.Popen([PY, *cmd.format(RUNS=RUNS).split()], cwd=SRC, stdout=log, stderr=subprocess.STDOUT,
                             env={**env_base, "CUDA_VISIBLE_DEVICES": str(g)})
        running[p] = g; results[i] = (cmd, p)
        print(f"[gpu {g}] started: {cmd}")
    for p in list(running):
        if p.poll() is not None:
            print(f"[gpu {running[p]}] exit {p.returncode}"); running.pop(p)
    time.sleep(30)
failed = [(c, p.returncode) for c, p in results.values() if p.returncode != 0]
print("FAILED:" if failed else "all commands finished", failed or "")

In [ ]:
# Summary of what will be saved as notebook output.
for d in sorted(os.listdir(RUNS)):
    if d.startswith("_"):
        continue
    full = os.path.join(RUNS, d)
    print(("complete   " if os.path.exists(f"{full}/metrics.json") else "INCOMPLETE ") + d)